# How big are New York's affordable housing buildings?

New York City's housing department (HPD) publishes a list of every building
that counts toward the city's affordable housing plans since 2014. The data is
on [NYC Open Data](https://data.cityofnewyork.us/d/hg8x-zxpr). Each row is one
building.

I want to know how many apartments a typical building has. I'll calculate the
mean, median and mode twice, first with pandas and then by hand, and finish
with a simple chart by borough.

In [1]:
import pandas as pd

In [2]:
housing = pd.read_csv("affordable_housing.csv")
housing.head()

,Project ID,Project Name,Project Start Date,Project Completion Date,Building ID,Number,Street,Borough,Postcode,BBL,...,2-BR Units,3-BR Units,4-BR Units,5-BR Units,6-BR+ Units,Unknown-BR Units,Counted Rental Units,Counted Homeownership Units,All Counted Units,Total Units
0,68872,177-06 WEXFORD TERRACE,06/30/2026,NaN,991637.0,177-06,WEXFORD TERRACE,Queens,11432.0,4.098350e+09,...,NaN,NaN,NaN,NaN,NaN,NaN,9.0,NaN,9.0,43
1,72436,ST. NICK'S. 819 GRAND STREET,06/30/2026,NaN,1021347.0,819,GRAND STREET,Brooklyn,11211.0,3.029220e+09,...,NaN,NaN,NaN,NaN,NaN,NaN,187.0,NaN,187.0,187
2,72449,BREMOND KING DAVIS.YR15.FY26,06/30/2026,NaN,21199.0,127,EDGECOMBE AVENUE,Manhattan,10030.0,1.020480e+09,...,20.0,NaN,NaN,NaN,NaN,NaN,25.0,NaN,25.0,25
3,72449,BREMOND KING DAVIS.YR15.FY26,06/30/2026,NaN,28145.0,602,ST NICHOLAS AVENUE,Manhattan,10030.0,1.020480e+09,...,5.0,NaN,NaN,NaN,NaN,NaN,16.0,NaN,16.0,16
4,72449,BREMOND KING DAVIS.YR15.FY26,06/30/2026,NaN,40555.0,129,WEST 133 STREET,Manhattan,10030.0,1.019180e+09,...,6.0,4.0,NaN,NaN,NaN,NaN,15.0,NaN,15.0,15


In [3]:
print(len(housing))
print(housing.columns)
print(housing["Total Units"].isna().sum())

9551
Index(['Project ID', 'Project Name', 'Project Start Date',
       'Project Completion Date', 'Building ID', 'Number', 'Street', 'Borough',
       'Postcode', 'BBL', 'BIN', 'Community Board', 'Council District',
       'Census Tract', 'NTA - Neighborhood Tabulation Area', 'Latitude',
       'Longitude', 'Latitude (Internal)', 'Longitude (Internal)',
       'Building Completion Date', 'Reporting Construction Type',
       'Extended Affordability Only', 'Prevailing Wage Status',
       'Extremely Low Income Units', 'Very Low Income Units',
       'Low Income Units', 'Moderate Income Units', 'Middle Income Units',
       'Other Income Units', 'Studio Units', '1-BR Units', '2-BR Units',
       '3-BR Units', '4-BR Units', '5-BR Units', '6-BR+ Units',
       'Unknown-BR Units', 'Counted Rental Units',
       'Counted Homeownership Units', 'All Counted Units', 'Total Units'],
      dtype='str')
0


In [4]:
units = housing["Total Units"]
units[~units.str.isdigit()]

1442    1,320
3406    1,046
4533    1,122
6687    1,000
6902    1,175
9040    1,028
9157    1,175
Name: Total Units, dtype: str

In [5]:
housing = pd.read_csv("affordable_housing.csv", thousands=",")
units = housing["Total Units"]

In [6]:
print(units.mean())
print(units.median())
print(units.mode()[0])

46.65354413150455
12.0
1


## The hard way: no pandas

Now I repeat the same three calculations using only what is built into Python,
so no pandas and no `statistics` module. First I read the file row by row with
the `csv` module and collect the number of units of every building in a plain
list. Seven buildings write their size with a comma, like `1,175`, so I remove
the comma before turning the text into a number.

In [7]:
import csv

unit_counts = []
with open("affordable_housing.csv") as file:
    reader = csv.DictReader(file)
    for row in reader:
        units_text = row["Total Units"]
        # Large buildings are written like '1,175', so drop the comma first.
        unit_counts.append(int(units_text.replace(",", "")))

print(len(unit_counts))
print(unit_counts[:5])

9551
[43, 187, 25, 16, 15]


In [8]:
def calculate_mean(numbers):
    """Mean of a list of numbers."""
    total = 0
    for number in numbers:
        total += number
    return total / len(numbers)


print(calculate_mean(unit_counts))

46.65354413150455


### Median

The median is the value in the middle once all buildings are sorted by size.
With an even number of buildings there is no single middle, so I take the
average of the two values around the centre.

In [9]:
def calculate_median(numbers):
    """Middle value of a list of numbers."""
    ordered = sorted(numbers)
    middle = len(ordered) // 2

    if len(ordered) % 2 == 1:
        return ordered[middle]
    else:
        return (ordered[middle - 1] + ordered[middle]) / 2


print(calculate_median(unit_counts))

12


In [10]:
def calculate_mode(numbers):
    """Most frequent value in a list of numbers."""
    counts = {}
    for number in numbers:
        if number in counts:
            counts[number] += 1
        else:
            counts[number] = 1

    most_common = None
    highest_count = 0
    for number, count in counts.items():
        if count > highest_count:
            most_common = number
            highest_count = count

    return most_common


print(calculate_mode(unit_counts))

1


## What the three numbers say

YOUR EXPLANATION HERE (2 to 3 sentences, see the questions below)

The projects that make headlines look like this one:

![Meme: a headline about a parking lot becoming 200 affordable housing units, with a sarcastic reply](housing_meme.jpg)

*Source: [Instagram](https://www.instagram.com/p/DcRoS7ASukK/), showing a post by @lukepbeasley.*

A project like that, with more than 200 units, is the exception. The typical building in the city's data has just 12 units, and it's a handful of giants like this that pull the average up to 47.

In [11]:
units_by_borough = {}
with open("affordable_housing.csv") as file:
    reader = csv.DictReader(file)
    for row in reader:
        borough = row["Borough"]
        building_units = int(row["Total Units"].replace(",", ""))
        if borough in units_by_borough:
            units_by_borough[borough] += building_units
        else:
            units_by_borough[borough] = building_units

print(units_by_borough)

{'Queens': 63338, 'Brooklyn': 143062, 'Manhattan': 108735, 'Bronx': 123413, 'Staten Island': 7040}


In [12]:
largest = max(units_by_borough.values())
bar_width = 30

print("Total units in affordable housing buildings, by borough")
for borough, count in units_by_borough.items():
    bar = "█" * round(count / largest * bar_width)
    print(f"{borough:<15}{bar} {count:,}")

Total units in affordable housing buildings, by borough
Queens         █████████████ 63,338
Brooklyn       ██████████████████████████████ 143,062
Manhattan      ███████████████████████ 108,735
Bronx          ██████████████████████████ 123,413
Staten Island  █ 7,040
